# Human Sleep Patterns & Stress — Extended Solution Notebook

**Project Goal:** Continue helping a friend analyze wearable fitness tracker sleep data (6 months of daily measurements). Confirm whether the proportion of nights with adequate sleep (≥ 7 hours) exceeds 75 % via a formal hypothesis test, explore patterns, and extend the analysis with alternate methods, extra practice problems, and a simulation section.

**Data:** 179 daily observations (1 Nov 2021 – 30 Apr 2022) containing Sleep Score, Hours of Sleep, REM %, Deep %, Heart-Rate-Below-Resting %, and sleep/wake times.

**How to use:** Run all cells top-to-bottom. Every computation prints its result so you can follow the logic.

## Project Flowchart

The diagram below shows the complete analysis pipeline from raw data to conclusions and deliverables.

![Human Sleep Flowchart](human_sleep_flowchart.png)


## 0. Setup & Imports

Import the required libraries and set a clean plotting style.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
%matplotlib inline

print("Libraries loaded successfully.")


## 1. Load & Explore the Data

Load the CSV, convert the DATE column, and perform a quick inspection.

In [ ]:
df = pd.read_csv('human_sleep_patterns.csv', parse_dates=['DATE'])
df = df.sort_values('DATE').reset_index(drop=True)

print("Shape:", df.shape)
print("\nDate range:", df['DATE'].min().date(), "→", df['DATE'].max().date())
print("\nColumns:", list(df.columns))
print("\nFirst 5 rows:")
print(df.head().to_string())
print("\nDescriptive statistics:")
print(df.describe().round(2).to_string())
print("\nMissing values:")
print(df.isnull().sum())


## 2. Exercise 1 — Hours of Sleep (Hypothesis Test for a Proportion)

It is known that adults need **at least 7 hours** of sleep per night for a healthy lifestyle. We will test whether the friend obtains ≥ 7 hours of sleep **at least 75 % of the time**.

### 2.1 Hypotheses & Test Type

- **H₀ (null):** p = 0.75
- **H₁ (alternative):** p > 0.75

Because the alternative is “greater than”, this is a **right-tailed** (one-sided) test.

We will use the normal approximation to the sampling distribution of p-hat (valid because n*p0 and n*(1-p0) are both > 5).


In [ ]:
# Parameters under H0
p0 = 0.75
alpha_05 = 0.05
alpha_10 = 0.10

# Step 2–4: count, n, p-hat
n = len(df)
count = (df['HOURS OF SLEEP'] >= 7).sum()
phat = count / n

print(f"Number of nights with ≥ 7 hours of sleep (count) = {count}")
print(f"Total nights in sample (n)                       = {n}")
print(f"Sample proportion (p-hat)                        = {phat:.6f}  ({phat*100:.2f} %)")


In [ ]:
# Step 5: test statistic
# z = (p-hat - p0) / sqrt( p0*(1-p0)/n )
se = np.sqrt(p0 * (1 - p0) / n)
z_stat = (phat - p0) / se
print(f"Standard error under H0 = {se:.6f}")
print(f"Test statistic z        = {z_stat:.6f}")


In [ ]:
# Step 6: p-value (right-tailed, normal)
p_value = 1 - stats.norm.cdf(z_stat)
print(f"p-value (right-tailed)  = {p_value:.6f}")


### 2.2 Decision

**Step 7:** Can we reject H₀ at the 5 % or 10 % significance level?


In [ ]:
print("Decision at α = 0.05:")
if p_value < alpha_05:
    print("  Reject H₀. Evidence that the true proportion > 0.75.")
else:
    print("  Fail to reject H₀. Insufficient evidence that the true proportion > 0.75.")

print("\nDecision at α = 0.10:")
if p_value < alpha_10:
    print("  Reject H₀. Evidence that the true proportion > 0.75.")
else:
    print("  Fail to reject H₀. Insufficient evidence that the true proportion > 0.75.")

print(f"\nConclusion: With p ≈ {p_value:.4f} we cannot reject the null hypothesis at either conventional level.")
print("Although ~78.2 % of nights meet the 7-hour guideline, the excess over 75 % is not statistically significant given the sample size.")


## 3. Alternate Code Paths to the Same Result

### 3.1 Using `statsmodels.stats.proportion.proportions_ztest`


In [ ]:
# Alternate 1 – statsmodels
z_sm, p_sm = proportions_ztest(count, n, value=p0, alternative='larger', prop_var=p0)
print(f"statsmodels z = {z_sm:.6f},  p-value = {p_sm:.6f}")


### 3.2 Exact Binomial Test (no normal approximation)


In [ ]:
# Alternate 2 – exact binomial (one-sided)
try:
    res = stats.binomtest(count, n=n, p=p0, alternative='greater')
    binom_p = res.pvalue
except AttributeError:
    binom_p = stats.binom_test(count, n=n, p=p0, alternative='greater')

print(f"Exact binomial p-value = {binom_p:.6f}")
print("(Very close to the normal-approximation p-value, confirming the approximation is adequate.)")


## 4. More Practice Problems

### 4.1 One-sample t-test: Is the mean hours of sleep significantly greater than 7?


In [ ]:
# H0: μ = 7   vs   H1: μ > 7
mean_hours = df['HOURS OF SLEEP'].mean()
t_stat, p_t = stats.ttest_1samp(df['HOURS OF SLEEP'], popmean=7, alternative='greater')
print(f"Sample mean hours = {mean_hours:.4f}")
print(f"t-statistic       = {t_stat:.4f}")
print(f"p-value (one-sided) = {p_t:.6e}")
print("→ Strong evidence that average sleep duration exceeds 7 hours.")


### 4.2 Proportion of high-quality nights (Sleep Score ≥ 85)


In [ ]:
# Test whether Sleep Score ≥ 85 on more than 50 % of nights
count_high = (df['SLEEP SCORE'] >= 85).sum()
phat_high = count_high / n
z_high, p_high = proportions_ztest(count_high, n, value=0.50, alternative='larger')
print(f"Nights with Sleep Score ≥ 85: {count_high} / {n}  (p-hat = {phat_high:.3f})")
print(f"z = {z_high:.3f},  p-value = {p_high:.4e}")
print("→ Highly significant: the friend achieves a high sleep score more than half the time.")


### 4.3 Correlation analysis


In [ ]:
corr_matrix = df[['SLEEP SCORE','HOURS OF SLEEP','REM SLEEP','DEEP SLEEP','HEART RATE BELOW RESTING']].corr()
print("Pearson correlation matrix:")
print(corr_matrix.round(3).to_string())

print("\nStrongest positive associations with Sleep Score:")
print(corr_matrix['SLEEP SCORE'].drop('SLEEP SCORE').sort_values(ascending=False).round(3).to_string())


## 5. Pattern Discovery & Additional Insights

### 5.1 Monthly trends


In [ ]:
df['month'] = df['DATE'].dt.to_period('M').astype(str)
monthly = df.groupby('month').agg(
    nights=('HOURS OF SLEEP', 'size'),
    nights_ge7=('HOURS OF SLEEP', lambda x: (x >= 7).sum()),
    mean_hours=('HOURS OF SLEEP', 'mean'),
    mean_score=('SLEEP SCORE', 'mean'),
    mean_rem=('REM SLEEP', 'mean'),
    mean_deep=('DEEP SLEEP', 'mean')
).reset_index()
monthly['prop_ge7'] = monthly['nights_ge7'] / monthly['nights']
print(monthly.round(3).to_string())

print("\nObservation: November 2021 and February 2022 show the lowest proportions of adequate-sleep nights.")


### 5.2 Visualizations


In [ ]:
from IPython.display import Image, display
print("Hours of Sleep distribution & boxplot:")
display(Image('sleep_hours_dist.png'))
print("\nSleep Score relationships:")
display(Image('sleep_score_scatter.png'))
print("\nMonthly proportion of adequate sleep:")
display(Image('sleep_monthly_prop.png'))
print("\nCorrelation heatmap:")
display(Image('sleep_corr_heatmap.png'))


## 6. Simulation Section

Modify key parameters and observe how the test result changes.

### 6.1 Sensitivity to the sleep-duration threshold


In [ ]:
def proportion_test(threshold=7.0, p0=0.75, alpha=0.05):
    """Run the one-sided proportion test for a chosen threshold."""
    count = (df['HOURS OF SLEEP'] >= threshold).sum()
    phat = count / n
    se = np.sqrt(p0 * (1 - p0) / n)
    z = (phat - p0) / se
    pval = 1 - stats.norm.cdf(z)
    reject = pval < alpha
    return {
        'threshold': threshold,
        'count': count,
        'phat': phat,
        'z': z,
        'pval': pval,
        'reject_H0': reject
    }

print("Varying the minimum-hours threshold (p0 fixed at 0.75, α=0.05):")
for thr in [6.5, 7.0, 7.5, 8.0]:
    r = proportion_test(threshold=thr)
    print(f"  ≥{thr}h → count={r['count']:3d}, p-hat={r['phat']:.3f}, z={r['z']:.2f}, p={r['pval']:.4f}, reject={r['reject_H0']}")


### 6.2 Sensitivity to the null proportion p0


In [ ]:
print("Varying the hypothesized proportion p0 (threshold=7h, α=0.05):")
for p0_val in [0.70, 0.75, 0.80, 0.85]:
    r = proportion_test(threshold=7.0, p0=p0_val)
    print(f"  p0={p0_val:.2f} → z={r['z']:.2f}, p={r['pval']:.4f}, reject={r['reject_H0']}")


### 6.3 Bootstrap confidence interval for the true proportion


In [ ]:
np.random.seed(42)
n_boot = 5000
boot_phats = []
for _ in range(n_boot):
    sample = df['HOURS OF SLEEP'].sample(n, replace=True)
    boot_phats.append((sample >= 7).mean())

ci_low, ci_high = np.percentile(boot_phats, [2.5, 97.5])
print(f"Bootstrap 95 % CI for p (nights ≥ 7 h): [{ci_low:.4f}, {ci_high:.4f}]")
print(f"Point estimate p-hat = {phat:.4f}")
print("Note that 0.75 lies inside the interval → consistent with failing to reject H0.")


### 6.4 Quick power sketch (under a true p = 0.80)


In [ ]:
# Approximate power of the current design if true p were 0.80
true_p = 0.80
z_crit = stats.norm.ppf(0.95)
se0 = np.sqrt(0.75*0.25/n)
threshold_phat = 0.75 + z_crit * se0
se_true = np.sqrt(true_p*(1-true_p)/n)
power = 1 - stats.norm.cdf(threshold_phat, loc=true_p, scale=se_true)
print(f"Approx. power to detect true p = 0.80 (α=0.05, n={n}): {power:.3f}")
print("(Moderate power; a larger sample or a larger effect would improve detection.)")


## 7. Key Takeaways

1. **Main test result:** 140 of 179 nights (78.2 %) had ≥ 7 hours of sleep. The one-sided z-test against p₀ = 0.75 yields z ≈ 0.99 and p ≈ 0.160. We **fail to reject H₀** at both the 5 % and 10 % levels.

2. **Practical interpretation:** The observed proportion is numerically higher than 75 %, but the difference is not statistically significant with the present sample size. The friend is doing reasonably well, yet we cannot claim with confidence that the long-run proportion exceeds three-quarters.

3. **Average duration is solid:** Mean sleep ≈ 7.56 h; a one-sided t-test against μ = 7 is highly significant.

4. **Sleep quality drivers:** Sleep Score correlates most strongly with “% Heart Rate Below Resting” (r ≈ 0.50) and moderately with REM % and hours of sleep.

5. **Temporal pattern:** November 2021 and February 2022 show dips in the proportion of adequate-sleep nights; other months are comfortably above 80 %.

6. **Simulation insight:** Raising the threshold to 7.5 h or raising p₀ to 0.80 quickly flips the decision. The 95 % bootstrap CI comfortably contains 0.75, reinforcing the main conclusion.
